# SDTM Variable Identity — Pass B (3a-bis)

**Purpose.** Extend `SDTM_Variable_Identity.xlsx` with NCIt identity for
COSMoS variables that are missing from the two primary subsets (CDISC
Variable Terminology / CDISC Root Variable Terminology).

**Why.** Pass A covered 338/449 COSMoS variables. The remaining 111 have
no NCIt identity in those two subsets. They fall into three groups:
- (a) present in a different CDISC Variable Terminology subset (CDASH, SEND)
- (b) present in Thesaurus but with no Variable Terminology subset flag at all
- (c) absent from Thesaurus as an ALL-CAPS synonym.

This notebook finds (a) and (b); (c) remain unresolved and are reported.

**Inputs**
- `sdtm-narrative/reference/SDTM_Variable_Identity.xlsx` — Pass A output
- `sdtm-test-codes/downloads/Thesaurus.txt`
- `cosmos-bc-dss/interim/COSMoS_Graph.xlsx`

**Output.** Same xlsx path — Pass A rows preserved, Pass B rows appended.
`Subset` column gets new enum values:
- `CDASH` — in CDISC CDASH Variable Terminology
- `SEND` — in CDISC SEND Variable Terminology
- `CDASH;SEND` — in both
- `Other` — in Thesaurus but in no CDISC Variable Terminology subset

**Disambiguation.** The variable code appears as an ALL-CAPS pipe-separated
synonym. Concepts are excluded if their preferred term looks like a codelist
container (starts with 'CDISC' and contains 'Terminology'). If multiple
concepts survive filtering, all are kept and flagged in `Notes`.


## Setup


In [1]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

REPO = Path.cwd().parent.parent
THES     = REPO / 'sdtm-test-codes'  / 'downloads'  / 'Thesaurus.txt'
GRAPH    = REPO / 'cosmos-bc-dss'    / 'interim'    / 'COSMoS_Graph.xlsx'
IDENTITY = REPO / 'sdtm-narrative'   / 'reference'  / 'SDTM_Variable_Identity.xlsx'

assert IDENTITY.exists(), f'Pass A output missing: {IDENTITY}'
print(f'Identity xlsx: {IDENTITY}')


Identity xlsx: /sessions/trusting-zen-bell/mnt/cdisc-for-ai/sdtm-narrative/reference/SDTM_Variable_Identity.xlsx


## Load Pass A identity + COSMoS variable universe


In [2]:
pass_a = pd.read_excel(IDENTITY, sheet_name='Variable_Identity', dtype=str, na_filter=False)
print(f'Pass A rows: {len(pass_a)}')

# COSMoS variables and their domains
graph_vars = pd.read_excel(GRAPH, sheet_name='Variables')[['ds_id', 'variable_name']]
graph_dss  = pd.read_excel(GRAPH, sheet_name='DSS')[['ds_id', 'domain']]
vars_with_domain = graph_vars.merge(graph_dss, on='ds_id', how='left')
cosmos_var_domains = (vars_with_domain
    .groupby('variable_name')['domain']
    .apply(lambda s: sorted(set(s.dropna())))
    .to_dict()
)
cosmos_variables = set(cosmos_var_domains.keys())

# Missing = in COSMoS but not covered by Pass A
pass_a_vars = set(pass_a.loc[pass_a['Variable'] != '', 'Variable'])
missing = sorted(cosmos_variables - pass_a_vars)
print(f'COSMoS variables:           {len(cosmos_variables)}')
print(f'Covered by Pass A:          {len(cosmos_variables & pass_a_vars)}')
print(f'Missing (Pass B targets):   {len(missing)}')


Pass A rows: 1470


COSMoS variables:           449
Covered by Pass A:          338
Missing (Pass B targets):   111


## Load Thesaurus and search


In [3]:
FLAT_COLS = ['code', 'url', 'parents', 'synonyms', 'definition',
             'display_name', 'concept_status', 'semantic_type', 'subset_membership']

flat = pd.read_csv(THES, sep='\t', header=None, names=FLAT_COLS, dtype=str, na_filter=False)
print(f'Thesaurus concepts: {len(flat):,}')

# Precompute: code -> list of ALL-CAPS synonyms (to search against)
VAR_CODE_RE = re.compile(r'^[A-Z0-9]{2,8}$')

def all_caps_syns(synonyms: str) -> set[str]:
    parts = [s.strip() for s in synonyms.split('|') if s.strip()]
    return {p for p in parts if VAR_CODE_RE.fullmatch(p)}

# Invert: ALL-CAPS token -> set of concept codes carrying it as a synonym
token_to_codes: dict[str, list[int]] = {}
for idx, syns in enumerate(flat['synonyms']):
    for tok in all_caps_syns(syns):
        token_to_codes.setdefault(tok, []).append(idx)
print(f'Distinct ALL-CAPS synonym tokens indexed: {len(token_to_codes):,}')


Thesaurus concepts: 211,072
Distinct ALL-CAPS synonym tokens indexed: 61,454


## Match each missing variable, apply disambiguation


In [4]:
CDASH_SUBSET = 'CDISC CDASH Variable Terminology'
SEND_SUBSET  = 'CDISC SEND Variable Terminology'
CODELIST_RE  = re.compile(r'^CDISC .* Terminology$')

def subset_label(membership: str) -> str:
    in_cdash = CDASH_SUBSET in membership
    in_send  = SEND_SUBSET  in membership
    if in_cdash and in_send: return 'CDASH;SEND'
    if in_cdash: return 'CDASH'
    if in_send:  return 'SEND'
    return 'Other'

pass_b_rows = []
unresolved = []

for var in missing:
    idxs = token_to_codes.get(var, [])
    hits = flat.iloc[idxs] if idxs else flat.iloc[0:0]

    # Exclude codelist-container concepts
    pt = hits['synonyms'].map(lambda s: s.split('|')[0].strip())
    keep = ~pt.str.match(CODELIST_RE, na=False)
    hits = hits[keep].copy()
    hits['preferred_term'] = hits['synonyms'].map(lambda s: s.split('|')[0].strip())

    if len(hits) == 0:
        unresolved.append(var)
        continue

    if len(hits) == 1:
        r = hits.iloc[0]
        pass_b_rows.append({
            'variable':           var,
            'nci_code':           r['code'],
            'natural_name':       r['preferred_term'],
            'definition':         r['definition'],
            'subset':             subset_label(r['subset_membership']),
            'applicable_domains': ';'.join(cosmos_var_domains.get(var, [])),
            'notes':              '',
        })
    else:
        # Multiple survive — prefer concepts in any CDISC-scoped subset
        cdisc_hits = hits[hits['subset_membership'].str.contains('CDISC ', regex=False)]
        if len(cdisc_hits) == 1:
            r = cdisc_hits.iloc[0]
            dropped = [c for c in hits['code'] if c != r['code']]
            pass_b_rows.append({
                'variable':           var,
                'nci_code':           r['code'],
                'natural_name':       r['preferred_term'],
                'definition':         r['definition'],
                'subset':             subset_label(r['subset_membership']),
                'applicable_domains': ';'.join(cosmos_var_domains.get(var, [])),
                'notes':              f'Multiple Thesaurus hits {list(hits["code"])}; chose CDISC-scoped {r["code"]} (dropped: {dropped})',
            })
        else:
            # Keep all remaining as separate rows, each flagged
            for _, r in hits.iterrows():
                pass_b_rows.append({
                    'variable':           var,
                    'nci_code':           r['code'],
                    'natural_name':       r['preferred_term'],
                    'definition':         r['definition'],
                    'subset':             subset_label(r['subset_membership']),
                    'applicable_domains': ';'.join(cosmos_var_domains.get(var, [])),
                    'notes':              f'Ambiguous: {len(hits)} Thesaurus concepts match synonym {var}; manual pick needed',
                })

pass_b = pd.DataFrame(pass_b_rows)
print(f'Pass B rows built:                    {len(pass_b)}')
print(f'Missing vars now resolved:            {pass_b["variable"].nunique() if len(pass_b) else 0}')
print(f'Missing vars still unresolved:        {len(unresolved)}')
print()
if len(pass_b):
    print('Subset distribution:')
    print(pass_b['subset'].value_counts().to_string())
    print()
    print('Flagged Pass B rows:')
    for _, r in pass_b[pass_b['notes'] != ''].iterrows():
        print(f'  {r["variable"]:10} {r["nci_code"]:10} {r["natural_name"][:40]:40} | {r["notes"][:80]}')

print()
print(f'Unresolved ({len(unresolved)}):')
print('  ' + ', '.join(unresolved))


Pass B rows built:                    9
Missing vars now resolved:            9
Missing vars still unresolved:        102

Subset distribution:
subset
Other    7
CDASH    2

Flagged Pass B rows:
  AGE        C25150     Age                                      | Multiple Thesaurus hits ['C106312', 'C25150']; chose CDISC-scoped C25150 (droppe
  RACE       C17049     Race                                     | Multiple Thesaurus hits ['C17049', 'C180698', 'C48210', 'C50840', 'C159096']; ch

Unresolved (102):
  BECAT, BEDECOD, BEPARTY, BESPEC, BETERM, CMRSDISC, CMTRTINT, CMTRTSTT, DDORRESU, DDSTRESN, DDSTRESU, ETHNIC, FALNKID, FAMETHOD, FTASTDEV, FTCAT, FTDTC, FTORRESU, FTREASND, FTSCAT, FTSTRESC, FTSTRESN, FTSTRESU, FTTEST, FTTESTCD, GFANMETH, GFCAT, GFCHROM, GFCOPYID, GFDTC, GFGENLOC, GFGENREF, GFGENSR, GFINHERT, GFMETHOD, GFORRES, GFORRESU, GFPRVID, GFSEQID, GFSPEC, GFSTRESC, GFSTRESN, GFSTRESU, GFSYM, GFSYMTYP, GFTEST, GFTESTCD, GFTSTDTL, ISBDAGNT, ISTSTDTL, MHLLT, MHLLTCD, MHPTCD, MKDT

## Merge Pass A + Pass B and rewrite xlsx

Pass A rows preserved byte-for-byte. Pass B rows appended. Same file path.
README sheet regenerated with updated counts and a new section documenting
Pass B.


In [5]:
pass_a_std = pass_a.rename(columns={
    'Variable':           'variable',
    'NCIt_Code':          'nci_code',
    'Natural_Name':       'natural_name',
    'Definition':         'definition',
    'Subset':             'subset',
    'Applicable_Domains': 'applicable_domains',
    'Notes':              'notes',
})[['variable', 'nci_code', 'natural_name', 'definition', 'subset', 'applicable_domains', 'notes']]

combined = pd.concat([pass_a_std, pass_b], ignore_index=True)

# Sort: empty variable last, otherwise by variable then nci_code
combined['_sort_key'] = combined['variable'].where(combined['variable'] != '', 'ZZZZ')
combined = combined.sort_values(['_sort_key', 'nci_code']).drop(columns='_sort_key').reset_index(drop=True)

print(f'Combined rows: {len(combined):,}')
print(f'  Pass A: {len(pass_a_std):,}')
print(f'  Pass B: {len(pass_b):,}')
print()
print('Subset value counts:')
print(combined['subset'].value_counts().to_string())


Combined rows: 1,479
  Pass A: 1,470
  Pass B: 9

Subset value counts:
subset
Variable         1296
Root              173
Other               7
CDASH               2
Variable+Root       1


In [6]:
HDR_FONT  = Font(name='Arial', bold=True, size=10, color='FFFFFF')
HDR_FILL  = PatternFill('solid', fgColor='1F4E79')
DATA_FONT = Font(name='Arial', size=9)
DATA_FILL = PatternFill('solid', fgColor='EEF2F7')
NOTE_FILL = PatternFill('solid', fgColor='FFF3CD')
THIN = Border(left=Side(style='thin', color='AAAAAA'),
              right=Side(style='thin', color='AAAAAA'),
              top=Side(style='thin', color='AAAAAA'),
              bottom=Side(style='thin', color='AAAAAA'))

COLUMNS = [
    ('variable',           'Variable',            14),
    ('nci_code',           'NCIt_Code',           12),
    ('natural_name',       'Natural_Name',        50),
    ('definition',         'Definition',          80),
    ('subset',             'Subset',              16),
    ('applicable_domains', 'Applicable_Domains',  30),
    ('notes',              'Notes',               80),
]

wb = Workbook()

ws_r = wb.active
ws_r.title = 'README'
ws_r.sheet_properties.tabColor = '1F4E79'

title_font   = Font(name='Arial', bold=True, size=14, color='1F4E79')
section_font = Font(name='Arial', bold=True, size=11, color='1F4E79')
body_font    = Font(name='Arial', size=10)

now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
n_total       = len(combined)
n_a           = len(pass_a_std)
n_b           = len(pass_b)
n_resolved    = (combined['variable'] != '').sum()
n_notes       = (combined['notes']    != '').sum()
n_with_doms   = (combined['applicable_domains'] != '').sum()
cosmos_cov    = (combined.loc[combined['applicable_domains'] != '', 'variable']).nunique()
subset_counts = combined['subset'].value_counts()

def subset_line(label):
    return f'  {label:18} {subset_counts.get(label, 0):>6,}'

readme = [
    ('SDTM VARIABLE IDENTITY', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    ('', None),
    (f'Generated:           {now}', body_font),
    ('Pipeline:            sdtm-narrative/notebooks/20_build_variable_identity.ipynb (Pass A)', body_font),
    ('                     sdtm-narrative/notebooks/22_variable_identity_bis.ipynb  (Pass B)', body_font),
    ('NCIt source:         Thesaurus.txt (NCI EVS FLAT file, versionless)', body_font),
    ('COSMoS source:       cosmos-bc-dss/interim/COSMoS_Graph.xlsx', body_font),
    ('', None),
    ('SCOPE', section_font),
    ('', None),
    (f'Total rows:          {n_total:,}', body_font),
    (f'  Pass A:            {n_a:,}  (CDISC Variable + Root Variable Terminology subsets)', body_font),
    (f'  Pass B:            {n_b:,}  (second-pass lookup for 111 missing COSMoS variables)', body_font),
    ('', None),
    ('Subset distribution:', body_font),
    (subset_line('Variable'),      body_font),
    (subset_line('Root'),          body_font),
    (subset_line('Variable+Root'), body_font),
    (subset_line('CDASH'),         body_font),
    (subset_line('SEND'),          body_font),
    (subset_line('CDASH;SEND'),    body_font),
    (subset_line('Other'),         body_font),
    ('', None),
    (f'COSMoS variable coverage:  {cosmos_cov}/{len(cosmos_variables)} = {100*cosmos_cov/len(cosmos_variables):.0f}%', body_font),
    (f'Flagged in Notes:          {n_notes}', body_font),
    ('', None),
    ('COLUMNS', section_font),
    ('', None),
    ('Variable           SDTM variable code. Root-subset variables use the two-dash prefix', body_font),
    ('                   form in SDTMIG usage (e.g. --DESC, --CAT); Thesaurus stores the bare', body_font),
    ('                   form (DESC, CAT).', body_font),
    ('NCIt_Code          NCIt concept C-code.', body_font),
    ('Natural_Name       NCIt preferred term.', body_font),
    ('Definition         NCIt definition.', body_font),
    ('Subset             CDISC subset the concept belongs to:', body_font),
    ('                     Variable / Root / Variable+Root  = Pass A primary subsets', body_font),
    ('                     CDASH / SEND / CDASH;SEND         = Pass B CDISC Variable Terminology family', body_font),
    ('                     Other                             = Pass B, no CDISC Variable Terminology subset', body_font),
    ('Applicable_Domains Semicolon-separated SDTM domain codes where the variable appears in', body_font),
    ('                   COSMoS Graph. Empty = not used in COSMoS.', body_font),
    ('Notes              Ambiguity flag. Empty = clean extraction.', body_font),
    ('', None),
    ('PASS B DESIGN', section_font),
    ('', None),
    ('Pass B was added because 111 COSMoS variables had no entry in the CDISC', body_font),
    ('Variable Terminology / Root Variable Terminology subsets. Pass B widens the', body_font),
    ('search to the full Thesaurus by ALL-CAPS synonym match, applies two filters:', body_font),
    ('  - Excludes concepts whose preferred term matches "CDISC * Terminology"', body_font),
    ('    (codelist containers, not variable concepts).', body_font),
    ('  - When multiple concepts survive, prefers CDISC-scoped concepts; if still', body_font),
    ('    ambiguous, keeps all rows and flags in Notes.', body_font),
    ('Remaining unresolved COSMoS variables are concepts absent from Thesaurus as', body_font),
    ('ALL-CAPS synonyms (e.g. ETHNIC, FTDTC, BECAT). They are not appended.', body_font),
    ('', None),
    ('HEADER COLOUR', section_font),
    ('', None),
    ('Navy #1F4E79 marks the Variable Identity layer. Distinct from green (TESTCD),', body_font),
    ('yellow (COSMoS), chocolate (Instrument C20993), copper (Container C211913),', body_font),
    ('and grey (keys).', body_font),
]

ws_r.column_dimensions['A'].width = 120
for i, (text, font) in enumerate(readme, start=1):
    cell = ws_r.cell(row=i, column=1, value=text)
    if font is not None:
        cell.font = font

ws = wb.create_sheet('Variable_Identity')
ws.sheet_properties.tabColor = '1F4E79'

for j, (field, header, width) in enumerate(COLUMNS, start=1):
    c = ws.cell(row=1, column=j, value=header)
    c.font = HDR_FONT
    c.fill = HDR_FILL
    c.alignment = Alignment(horizontal='left', vertical='center')
    c.border = THIN
    ws.column_dimensions[get_column_letter(j)].width = width

for i, row in enumerate(combined.itertuples(index=False), start=2):
    for j, (field, _, _) in enumerate(COLUMNS, start=1):
        value = getattr(row, field)
        c = ws.cell(row=i, column=j, value=value)
        c.font = DATA_FONT
        c.border = THIN
        c.alignment = Alignment(horizontal='left', vertical='top', wrap_text=(field in ('definition', 'notes')))
        c.fill = NOTE_FILL if (field == 'notes' and value) else DATA_FILL

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

wb.save(IDENTITY)
print(f'Wrote {IDENTITY}')
print(f'Size: {IDENTITY.stat().st_size / 1024:.0f} KB')


Wrote /sessions/trusting-zen-bell/mnt/cdisc-for-ai/sdtm-narrative/reference/SDTM_Variable_Identity.xlsx
Size: 96 KB


## Sanity checks


In [7]:
assert combined['nci_code'].is_unique, 'duplicate nci_code'
print('OK: nci_code unique')

still_missing = cosmos_variables - set(combined.loc[combined['variable'] != '', 'variable'])
print(f'COSMoS variables still missing from identity: {len(still_missing)}')
if still_missing:
    for v in sorted(still_missing):
        print(f'  {v}')


OK: nci_code unique
COSMoS variables still missing from identity: 102
  BECAT
  BEDECOD
  BEPARTY
  BESPEC
  BETERM
  CMRSDISC
  CMTRTINT
  CMTRTSTT
  DDORRESU
  DDSTRESN
  DDSTRESU
  ETHNIC
  FALNKID
  FAMETHOD
  FTASTDEV
  FTCAT
  FTDTC
  FTORRESU
  FTREASND
  FTSCAT
  FTSTRESC
  FTSTRESN
  FTSTRESU
  FTTEST
  FTTESTCD
  GFANMETH
  GFCAT
  GFCHROM
  GFCOPYID
  GFDTC
  GFGENLOC
  GFGENREF
  GFGENSR
  GFINHERT
  GFMETHOD
  GFORRES
  GFORRESU
  GFPRVID
  GFSEQID
  GFSPEC
  GFSTRESC
  GFSTRESN
  GFSTRESU
  GFSYM
  GFSYMTYP
  GFTEST
  GFTESTCD
  GFTSTDTL
  ISBDAGNT
  ISTSTDTL
  MHLLT
  MHLLTCD
  MHPTCD
  MKDTC
  MKEVAL
  MKEVALID
  MKLAT
  MKLOC
  MKMETHOD
  MKORRES
  MKORRESU
  MKSTRESC
  MKSTRESN
  MKSTRESU
  MKTEST
  MKTESTCD
  PRTRTINT
  PRTRTSTT
  PRVIR
  PRVIRP
  QSANTXHI
  QSANTXLO
  QSANVLHI
  QSANVLLO
  QSEVINTX
  QSMETHOD
  RPLAT
  RPLOC
  RSEVINTX
  RSEVLINT
  RSLOC
  RSORRESU
  RSSCAT
  RSSTRESN
  TRREASNE
  TULOCDTL
  TUPRVIR
  TUPRVIRP
  URCAT
  URDIR
  URDTC
  URLAT
  URLOC